In [1]:
import torch
! pip install tiktoken


   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/940.9 kB ? eta -:--:--
   ---


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import importlib

In [3]:
import tiktoken

In [4]:
print(importlib.metadata.version("tiktoken"))

0.14.0


#### tiktoken is also used by openai for BPE

In [5]:
tokenizer=tiktoken.get_encoding("gpt2")

In [16]:
text=(
    "hello, do you like tea? <|endoftext|> in the sunlit terraces "
    "of someunknownPlace."
)

In [17]:
integer=tokenizer.encode(text,allowed_special={"<|endoftext|>"})

In [18]:
print(integer)

[31373, 11, 466, 345, 588, 8887, 30, 220, 50256, 287, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [19]:
string=tokenizer.decode(integer)

In [20]:
print(string)

hello, do you like tea? <|endoftext|> in the sunlit terraces of someunknownPlace.


#### if it was word based tokenizer it would have thrown an error for "someunknownPlace" as it would have not been in its dictionary or vocab while if it was charachter based tokenizer it would have lost the context or meaning of words.
### byte pair or sub_word based tokenizer solves both the probluems



# Creating Input-Target pairs

#### using sliding window approach

In [44]:
with open("verdict.txt","r",encoding="utf-8") as f:
    text=f.read()

In [45]:
encript=tokenizer.encode(text)

In [46]:
print(encript)

[10970, 33310, 35, 18379, 198, 198, 15749, 40417, 198, 198, 40, 550, 1464, 1807, 3619, 402, 271, 10899, 2138, 257, 7026, 15632, 438, 2016, 257, 198, 198, 11274, 5891, 1576, 438, 568, 340, 373, 645, 1049, 5975, 284, 502, 284, 3285, 198, 198, 5562, 11, 287, 262, 6001, 286, 465, 13476, 11, 339, 550, 5710, 465, 12036, 11, 198, 198, 30526, 257, 5527, 27075, 11, 290, 4920, 2241, 287, 257, 4489, 64, 319, 262, 198, 198, 49, 452, 41976, 13, 357, 10915, 314, 2138, 1807, 340, 561, 423, 587, 10598, 393, 198, 198, 7414, 382, 1198, 2014, 198, 198, 1, 464, 6001, 286, 465, 13476, 1, 438, 5562, 373, 644, 262, 1466, 1444, 340, 13, 314, 198, 198, 5171, 3285, 9074, 13, 46606, 536, 5469, 438, 14363, 938, 4842, 1650, 353, 438, 2934, 489, 3255, 198, 198, 14363, 48422, 540, 450, 67, 3299, 13, 366, 5189, 1781, 340, 338, 1016, 284, 3758, 262, 198, 198, 8367, 286, 616, 4286, 705, 1014, 510, 26, 475, 314, 836, 470, 892, 286, 326, 11, 1770, 13, 198, 198, 33048, 2763, 438, 1169, 2994, 284, 943, 17034, 318, 477, 314

In [47]:
enc_sample=encript[50:]

#### i am taking context size 4 which means if model gets 4 words as input it could predict the 5th word
#### for this pour purpose i am using simple sliding window approach where if the input belongs to x then output will be y
##### the input x is the first 4 tokens [1,2,3,4], and the target y is the next 4 tokens [2,3,4,5]

In [48]:
context=4
x=enc_sample[:context]
y=enc_sample[1:context+1]
print(x)
print(y)


[13476, 11, 339, 550]
[11, 339, 550, 5710]


In [49]:
for i in range(1,context+1):
    x=enc_sample[:i]
    y=enc_sample[i]

    print(x,"--->",y)

[13476] ---> 11
[13476, 11] ---> 339
[13476, 11, 339] ---> 550
[13476, 11, 339, 550] ---> 5710


In [51]:
for i in range(1,context+1):
    x=enc_sample[:i]
    desired=enc_sample[i]

    print(tokenizer.decode(x),"---->",tokenizer.decode([desired]))

 glory ----> ,
 glory, ---->  he
 glory, he ---->  had
 glory, he had ---->  dropped


##### I've now created the input-target pairs that we can turn into use for the LLM training

## implementing a Data Loader

#### for the efficient data loader implementation , we will use PyTorch's built-in Dataset and Dataloader classes

#### step1: tokenize the entire text
#### step2: use sliding window to chunk the data into overlaping sequence of max _length
#### step3: return the total number of rows in dataset
#### step4: return a single row from the dataset

In [52]:
from torch.utils.data import Dataset,DataLoader

class tokenizerv1(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids=[]
        self.target_ids=[]
        token_ids=tokenizer.encode(txt,allowed_special={"<|endoftext|>"})
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i+max_length]
            target_chunk=token_ids[i+1:max_length+1+i]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    def __len__(self):
        return len(self.input_ids)
    def __getitem__(self, ind):
        return self.input_ids[ind],self.target_ids[ind]


#### step1: initialize the tokenizer
#### step2: Create dataset
#### step3: drop_last=True drops the last batch if it is shorter than the specifies batch_size to prevent loss spike during training
#### step4: the number of CPU processes to use preprocessing

In [53]:
def crearte_dataloader_v1(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
    tokenizer=tiktoken.get_encoding("gpt2")
    dataset=tokenizerv1(txt,tokenizer,max_length,stride)

    dataloader=DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)
    return dataloader

In [54]:
with open("verdict.txt","r",encoding="utf-8") as f:
    raw_text=f.read()

In [55]:
import torch
dataloader=crearte_dataloader_v1(raw_text,batch_size=1,max_length=4,stride=1,shuffle=False)

data_iter=iter(dataloader)
first_batch=next(data_iter)
print(first_batch)

[tensor([[10970, 33310,    35, 18379]]), tensor([[33310,    35, 18379,   198]])]


In [59]:
import torch
dataloader=crearte_dataloader_v1(raw_text,batch_size=10,max_length=4,stride=1,shuffle=False)

data_iter=iter(dataloader)
x,y=next(data_iter)
print("x=",x,"\n","y=",y)

x= tensor([[10970, 33310,    35, 18379],
        [33310,    35, 18379,   198],
        [   35, 18379,   198,   198],
        [18379,   198,   198, 15749],
        [  198,   198, 15749, 40417],
        [  198, 15749, 40417,   198],
        [15749, 40417,   198,   198],
        [40417,   198,   198,    40],
        [  198,   198,    40,   550],
        [  198,    40,   550,  1464]]) 
 y= tensor([[33310,    35, 18379,   198],
        [   35, 18379,   198,   198],
        [18379,   198,   198, 15749],
        [  198,   198, 15749, 40417],
        [  198, 15749, 40417,   198],
        [15749, 40417,   198,   198],
        [40417,   198,   198,    40],
        [  198,   198,    40,   550],
        [  198,    40,   550,  1464],
        [   40,   550,  1464,  1807]])
